# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RamyaMuthumariappan/FlyRankAI/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess, json, warnings
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/RamyaMuthumariappan/FlyRankAI.git"
REPO_DIR = "FlyRankAI"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found - are you at the repo root?"

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, GroupKFold, GroupShuffleSplit
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance
pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 30)

SEED = 42
RECEIPTS = {}          # every headline number in this notebook is stored here and written to work/outputs/
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Working dir:", os.getcwd())
print("Rows, columns:", df.shape, "| distinct clients:", df["client_id"].nunique(), "| seed:", SEED)

Working dir: /content/FlyRankAI
Rows, columns: (30000, 44) | distinct clients: 32 | seed: 42


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

First, what the paper already does well, because it is the standard I am measuring my own work against: it states an evidence standard up front, tags results as CONFIRMED / NUANCED / REVERSED, demotes the unstable `361+` freshness bucket on its own, and says outright that the health-score feature ranking is "descriptive rather than causal." The questions below are the ones I would want a reviewer to ask of *my* work. I picked these two because they test two different links in the evidence chain: an **outcome** built from other columns, and a **validation design**.

### Finding A - Finding #4, "The Freshness Multiplier"
*Paper's statement: refreshing 365+ day pages within 30 days is associated with health 10.7 -> 34.5 (3.2x) and impressions 71 -> 4,039 (57x).*

- **Where does the outcome come from?** Health is a composite - impressions (30 pts) + position (30) + CTR (20) + scroll (20) - so 60 of 100 points are search visibility. If impressions rise 57x, health moves almost by construction. Question: what does the picture look like on clicks or position alone, outside the composite?
- **Where does "refreshed" come from?** From a last-updated date. Could pages be edited *because* they had already started to pick up traffic (the arrow running the other way)? Question: is there any information about *why* each page was updated?
- **Does the validation design carry "lever"?** The comparison reads as two different groups of pages in one snapshot, not the same pages before and after. Question: what are the group sizes, and what do medians look like? Impressions are heavy-tailed, so a mean of 4,039 vs 71 can be carried by a small number of pages. Matching on brand, topic and age would help too.
- **A consistency question, asked in good faith:** Finding #8's heatmap shows 44.6 health for `365+ x 0-30d fresh`, while Finding #4 quotes 34.5 for refreshed 365+ pages. Are these different filters or cohorts? A one-line note would let a reader reconcile them.
- **What already helps:** the paper's own wording ("in this portfolio", "strongest measured levers") is observational. A same-page before/after would let that read as an association with a design behind it.

### Finding B - ML appendix, "What Predicts Growth?"
*Paper's statement: a logistic regression separating growing from declining pages reaches 71% holdout accuracy (80/20 split, 61.8K pieces, 57 brands).*

- **Where does the label come from?** Growing/declining is defined from the change in impressions, last 30 days vs the previous 30. The appendix lists impressions, clicks and days visible among the inputs and calls "recent impressions" one of the strongest signals. Question: which impressions window is that? If it is the last-30-day count, it is one of the two numbers the label is computed from.
- **Does the split carry the claim?** An 80/20 split of pages from 57 brands - by row, or by brand? Pages from one brand share hidden character (topic, site authority, tracking setup), so a row split can reward memorising the brand. Question: how does accuracy change when whole brands are held out? This is exactly the weakness I test on my own model in Section 2.
- **What is the base rate?** 71% needs a naive baseline beside it. If the appendix mix resembles Finding #1 (74.8K up vs 45.6K down), always guessing "up" scores about 62%, so the model is roughly 9 points above naive (this is an assumption about the mix, not a measured fact). The appendix already labels itself exploratory and descriptive; printing the base rate next to the 71% would let readers calibrate at a glance.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Arithmetic behind the two methodology questions. Inputs are figures printed in the paper itself.

# --- Finding #4 (Freshness Multiplier): "3.2x health, 57x impressions" for 365+ day pages refreshed within 30 days
health_before, health_after = 10.7, 34.5
imp_before, imp_after = 71, 4039
print(f"Finding #4 ratios recomputed from the paper's own numbers: health x{health_after/health_before:.2f}, impressions x{imp_after/imp_before:.1f}")

# The paper defines Health = impressions (30 pts) + position (30) + CTR (20) + scroll depth (20).
weights = {"impressions": 30, "position": 30, "ctr": 20, "scroll": 20}
vis = weights["impressions"] + weights["position"]
print(f"Share of the Health score that is search-visibility (impressions + position): {vis}/{sum(weights.values())} = {vis/100:.0%}")
print("   -> an outcome measured partly in impressions will move when impressions move, whatever the cause.")

# --- ML appendix (growth vs decline logistic regression, '71% holdout accuracy')
# Finding #1's direction table gives the up/down mix. IF the appendix sample resembles it, the naive majority-class accuracy is:
up, down = 74.8, 45.6   # thousands of pages, Finding #1 table
majority = up / (up + down)
print(f"\nMajority-class accuracy if the appendix mix resembles Finding #1: {majority:.3f}")
print(f"Reported 71% is about {71 - 100*majority:+.1f} points above that naive baseline (an assumption about the mix, not a measured fact).")

# A reminder that I hold my OWN work to the same test: the base rate of my W5 target
lab = (df["trend_direction"] == "down")
p1 = (df["avg_position"] > 0) & (df["avg_position"] <= 10) & lab
sv = (df["days_since_last_update"] > 60) & (df["impressions_last_30d"] > 50)
df["y_high"] = (p1 | sv).astype(int)
print(f"\nMy W5 target 'y_high' base rate: {df['y_high'].mean():.3f} (n={len(df):,}) -> any score of mine must be read against this")


Finding #4 ratios recomputed from the paper's own numbers: health x3.22, impressions x56.9
Share of the Health score that is search-visibility (impressions + position): 60/100 = 60%
   -> an outcome measured partly in impressions will move when impressions move, whatever the cause.

Majority-class accuracy if the appendix mix resembles Finding #1: 0.621
Reported 71% is about +8.9 points above that naive baseline (an assumption about the mix, not a measured fact).

My W5 target 'y_high' base rate: 0.423 (n=30,000) -> any score of mine must be read against this


**What the check cell shows.** Recomputed from the paper's own printed numbers: the 3.2x and 57x ratios reproduce; 60% of the health composite is visibility; and the 71% sits about 9 points above a 62% majority baseline *if* the mix matches Finding #1. None of that says the paper is wrong - it says which extra number (a base rate, a group size, a brand-level split) would let a reader trust each claim further.

I asked the same three questions of my own Week-5 work, and the sections below report what I found: where my label comes from (Section 3), whether my split carries my claim (Section 2), and what my base rate is (0.42).

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**What W5 already did:** one client-grouped 75/25 split (8 test clients, seed 42). That was the right design, but it leaves two things unshown: (1) how much a *naive* split would have flattered the model, and (2) how stable the score is across *different* sets of held-out clients, since one draw of 8 clients is a coarse test.

**What I do here:** same features, same Random Forest, same seed, three evaluations -
- **BEFORE** - random 5-fold: rows are shuffled, so every client appears on both sides of the split.
- **W5 as reported** - the single grouped split, for continuity.
- **AFTER** - client-grouped 5-fold (`GroupKFold`): each client is held out exactly once, every row is scored by a model that never saw its client, and I report the spread across folds.

A **time-aware** split is not possible here: the starter CSV is a single trailing-90-day snapshot with no dated rows (the code confirms no date column). Grouped-by-client is therefore the honest option, and it matches the question a new client would ask: "does this work on a client it never saw?" Precision@top-10% sits next to the base rate throughout.

In [3]:
# ---- rebuild the W5 feature set EXACTLY (same columns, same fills, same guard) ----
for c_out, c_in in [("log_impressions_90d","impressions_90d"),("log_clicks_90d","clicks_90d"),
                    ("log_sessions_90d","sessions_90d"),("log_ai_sessions_90d","ai_sessions_90d")]:
    df[c_out] = np.log1p(df[c_in])
df["has_word_count"] = df["word_count"].notna().astype(int)
df["has_keyword_data"] = df["search_volume"].notna().astype(int)

NUM = ["search_volume","competition","cpc","word_count","char_count","log_impressions_90d","log_clicks_90d",
       "log_sessions_90d","log_ai_sessions_90d","days_with_impressions","days_with_sessions","content_age_days",
       "days_since_last_update","ctr","avg_position","engagement_rate","scroll_rate","ai_traffic_pct",
       "has_word_count","has_keyword_data"]
CAT = ["competition_level","content_type","main_intent","age_tier","freshness_tier","word_count_tier",
       "impression_tier","position_tier"]
for c in NUM: df[c] = df[c].fillna(0)
for c in CAT: df[c] = df[c].fillna("unknown").astype(str)
assert {"trend_direction","trend_pct","is_declining_label","priority_tier","y_high"}.isdisjoint(NUM + CAT)

y = df["y_high"].values
groups = df["client_id"].values

def make_model(nums, cats):
    pre = ColumnTransformer([("num", StandardScaler(), nums),
                             ("cat", OneHotEncoder(handle_unknown="ignore"), cats)])
    rf = RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=20, random_state=SEED, n_jobs=-1)
    return Pipeline([("pre", pre), ("clf", rf)])

def prec_at_frac(scores, y_true, frac=0.10):
    k = max(1, int(round(frac * len(scores))))
    return float(y_true[np.argsort(-scores)[:k]].mean())

def cv_eval(nums, cats, splitter, use_groups):
    """Out-of-fold evaluation. Every row is scored by a model that never trained on it."""
    X = df[nums + cats]
    oof = np.full(len(df), np.nan)
    rows = []
    for i, (tr, te) in enumerate(splitter.split(X, y, groups if use_groups else None), 1):
        m = make_model(nums, cats).fit(X.iloc[tr], y[tr])
        s = m.predict_proba(X.iloc[te])[:, 1]
        oof[te] = s
        rows.append({"fold": i, "test_rows": len(te), "test_clients": len(set(groups[te])),
                     "clients_shared_with_train": len(set(groups[te]) & set(groups[tr])),
                     "base_rate": y[te].mean(), "auc": roc_auc_score(y[te], s),
                     "p_at_top10pct": prec_at_frac(s, y[te])})
    f = pd.DataFrame(rows)
    f["lift_top10pct"] = f["p_at_top10pct"] / f["base_rate"]
    return oof, f

# BEFORE: naive random row split      AFTER: client-grouped split (whole clients held out)
oof_rand, f_rand = cv_eval(NUM, CAT, StratifiedKFold(5, shuffle=True, random_state=SEED), use_groups=False)
oof_grp,  f_grp  = cv_eval(NUM, CAT, GroupKFold(5), use_groups=True)

print("Per-fold detail - CLIENT-GROUPED 5-fold (the honest split):")
print(f_grp.round(3).to_string(index=False))
print("\nPer-fold detail - RANDOM 5-fold (the naive split), same model:")
print(f_rand.round(3).to_string(index=False))

def summarize(name, f, oof):
    return {"split": name,
            "AUC (mean of folds)": round(f["auc"].mean(), 3),
            "AUC fold sd": round(f["auc"].std(ddof=0), 3),
            "AUC worst fold": round(f["auc"].min(), 3),
            "P@top-10% (mean)": round(f["p_at_top10pct"].mean(), 3),
            "base rate": round(y.mean(), 3),
            "lift @top-10%": round(f["lift_top10pct"].mean(), 2),
            "AUC pooled OOF": round(roc_auc_score(y, oof), 3)}

# Continuity with W5: its single 75/25 grouped split (8 test clients), same seed
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED).split(df, y, groups))
m5 = make_model(NUM, CAT).fit(df.iloc[tr][NUM + CAT], y[tr])
s5 = m5.predict_proba(df.iloc[te][NUM + CAT])[:, 1]
w5_row = {"split": "W5 as reported: ONE grouped 75/25 split (8 test clients)",
          "AUC (mean of folds)": round(roc_auc_score(y[te], s5), 3), "AUC fold sd": np.nan, "AUC worst fold": np.nan,
          "P@top-10% (mean)": round(prec_at_frac(s5, y[te]), 3), "base rate": round(y[te].mean(), 3),
          "lift @top-10%": round(prec_at_frac(s5, y[te]) / y[te].mean(), 2), "AUC pooled OOF": np.nan}

before_after = pd.DataFrame([summarize("BEFORE: random 5-fold (rows shuffled across clients)", f_rand, oof_rand),
                             w5_row,
                             summarize("AFTER: client-grouped 5-fold (all 32 clients, out-of-fold)", f_grp, oof_grp)])
print("\n=== BEFORE / AFTER (same features, same Random Forest, same seed) ===")
print(before_after.to_string(index=False))

# Spread across clients: is the score even across the catalogue, or carried by a few clients?
per_client = []
for c in np.unique(groups):
    idx = np.where(groups == c)[0]
    if len(idx) >= 200 and 0 < y[idx].sum() < len(idx):
        per_client.append(roc_auc_score(y[idx], oof_grp[idx]))
per_client = np.array(per_client)
print(f"\nPer-client AUC (grouped OOF, clients with >=200 rows, n={len(per_client)}): "
      f"min {per_client.min():.3f} | median {np.median(per_client):.3f} | max {per_client.max():.3f}")
print("Time-aware split available?", "report_date" in df.columns or "date" in df.columns,
      "-> the starter CSV is ONE trailing-90-day snapshot with no dated rows, so a grouped split is the honest option here.")

RECEIPTS["section2"] = {"random_5fold": summarize("random", f_rand, oof_rand),
                        "client_grouped_5fold": summarize("grouped", f_grp, oof_grp),
                        "w5_single_grouped_split": {k: (None if (isinstance(v, float) and np.isnan(v)) else v) for k, v in w5_row.items()},
                        "per_client_auc": {"n": int(len(per_client)), "min": round(float(per_client.min()), 3),
                                           "median": round(float(np.median(per_client)), 3), "max": round(float(per_client.max()), 3)}}


Per-fold detail - CLIENT-GROUPED 5-fold (the honest split):
 fold  test_rows  test_clients  clients_shared_with_train  base_rate   auc  p_at_top10pct  lift_top10pct
    1       7008             1                          0      0.543 0.964          1.000          1.843
    2       5731             7                          0      0.626 0.978          1.000          1.598
    3       5753             8                          0      0.244 0.942          0.958          3.921
    4       5755             8                          0      0.324 0.929          0.991          3.056
    5       5753             8                          0      0.353 0.915          0.885          2.507

Per-fold detail - RANDOM 5-fold (the naive split), same model:
 fold  test_rows  test_clients  clients_shared_with_train  base_rate   auc  p_at_top10pct  lift_top10pct
    1       6000            31                         31      0.423 0.962            1.0          2.364
    2       6000            31      

**Reading the table (measured, directional).**

- **The split type mattered less than I expected.** Random 5-fold AUC is 0.960 and client-grouped is 0.946 - a gap of about 0.014. Client memorisation is a small contributor for this model; I should not present "switching to a grouped split" as the fix. The bigger issue shows up in Section 3.
- **The score varies with which clients are held out.** Fold AUC ranges from about 0.92 to 0.98. Fold 1 is a single very large client held out on its own (7,008 rows, AUC 0.964). Across the 21 clients with at least 200 rows, per-client AUC runs from 0.66 to 0.99 (median 0.93), so at least one client is scored noticeably worse than the average suggests.
- **W5's single-split number sits at the low edge of that range** (AUC 0.910, 8 test clients, base rate 0.364). One draw of 8 clients cannot distinguish "the model is weaker" from "these clients were harder".
- **Precision@top-10% is close to its ceiling** (mean 0.97, fold range 0.89-1.00, against a 0.42 base rate), so it cannot separate good from very good here; AUC and the per-fold spread carry more information. Lift is also hard to compare across folds because fold base rates differ (0.24 to 0.63).

Before/after in one line: **random 0.960 -> grouped 0.946 AUC (all 32 clients, out-of-fold), with a fold range of roughly 0.92-0.98**.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

I use the three-type taxonomy from the validation skill and run each check on the exact 28 features the W5 model used (20 numeric, 8 categorical):

1. **Label-derived features** - was the label computed from a column that is also a feature?
2. **Future / overlapping windows** - does a feature's window contain the label's window?
3. **Decision-derived features** - product flags or scores from an existing system used as inputs?

Plus the housekeeping items: population selection, a planted-leak test of my own harness, a top-feature sanity check, and real failure examples.

For reference, the W5 target is `y_high = page1_decay OR stale_visible`, where `page1_decay = (0 < avg_position <= 10) AND trend_direction == "down"` and `stale_visible = days_since_last_update > 60 AND impressions_last_30d > 50`.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---- 3a. Timeline / window audit: what does each feature group know, and when? ----
audit = pd.DataFrame([
    ["avg_position, position_tier",           "trailing 90d",  "INGREDIENT of the label (page1_decay uses 0 < avg_position <= 10)", "label-derived (type 1)"],
    ["days_since_last_update, freshness_tier","as of export",   "INGREDIENT of the label (stale_visible uses days_since_last_update > 60)", "label-derived (type 1)"],
    ["impressions_90d (log), days_with_impressions, clicks_90d (log), impression_tier", "trailing 90d (contains the last 30d)",
                                              "window CONTAINS the label's impressions_last_30d window", "overlapping window (type 2)"],
    ["ctr, engagement_rate, scroll_rate, sessions, ai_*", "trailing 90d", "same 90d window; not a direct label ingredient", "overlap possible, weaker"],
    ["search_volume, competition, cpc, content_type, main_intent, word_count, content_age_days", "static metadata", "known before any window", "clean"],
    ["trend_direction, trend_pct, is_declining_label", "last30 vs prev30", "define the label", "BANNED - guard asserts absent"],
], columns=["feature group", "window", "relationship to y_high", "verdict"])
print(audit.to_string(index=False))

# how much of the 90d impressions sit in the label's own 30d window?
share_30 = (df["impressions_last_30d"] / df["impressions_90d"].replace(0, np.nan)).median()
rho = df[["impressions_last_30d", "impressions_90d"]].corr(method="spearman").iloc[0, 1]
print(f"\nSpearman(impressions_last_30d, impressions_90d) = {rho:.3f}; median share of 90d impressions inside last 30d = {share_30:.2f}")

# ---- 3b. Decision-derived features (product flags / existing-system scores)? ----
flag_like = [c for c in NUM + CAT if any(t in c.lower() for t in ["health", "flag", "score", "priority", "queue", "zombie"])]
print("\nProduct-flag / existing-score columns among features:", flag_like if flag_like else "none")

# ---- 3c. Population selection: does who is IN the table depend on the outcome window? ----
print(f"\nPopulation: min impressions_90d = {df['impressions_90d'].min():.0f}, min content_age_days = {df['content_age_days'].min():.0f}; "
      f"rows with impressions_last_30d == 0: {(df['impressions_last_30d'] == 0).mean():.1%}")
print("-> rows are kept only if they had >=1 impression in the trailing 90 days: that is a choice that touches the outcome window (disclosed in section 4).")

# ---- 3d. How much of the label is a plain function of two ordinary columns? ----
print(f"\nShare of y_high=1 rows explained by stale_visible alone (no trend needed): {(sv & (df['y_high'] == 1)).sum() / y.sum():.1%}")
decline_blind = ((df["avg_position"] > 0) & (df["avg_position"] <= 10)) | sv
print(f"A decline-BLIND rule (position 1-10 OR stale_visible): precision {y[decline_blind].mean():.3f}, recall {(decline_blind & (y == 1)).sum() / y.sum():.3f}")

RECEIPTS["section3_structure"] = {"spearman_imp30_vs_imp90": round(float(rho), 3),
                                  "median_share_of_90d_in_last30d": round(float(share_30), 3),
                                  "share_of_highs_from_stale_visible_alone": round(float((sv & (df['y_high'] == 1)).sum() / y.sum()), 3),
                                  "decline_blind_rule_precision": round(float(y[decline_blind].mean()), 3),
                                  "product_flag_columns_in_features": flag_like}


                                                                           feature group                               window                                                   relationship to y_high                       verdict
                                                             avg_position, position_tier                         trailing 90d        INGREDIENT of the label (page1_decay uses 0 < avg_position <= 10)        label-derived (type 1)
                                                  days_since_last_update, freshness_tier                         as of export INGREDIENT of the label (stale_visible uses days_since_last_update > 60)        label-derived (type 1)
         impressions_90d (log), days_with_impressions, clicks_90d (log), impression_tier trailing 90d (contains the last 30d)                  window CONTAINS the label's impressions_last_30d window   overlapping window (type 2)
                                       ctr, engagement_rate, scroll_rate, sessions, 

**Audit findings so far (measured).**

1. **Label-derived (type 1) - found.** Four of my 28 features are direct ingredients of the label: `avg_position` and `position_tier` (page-1 decay), `days_since_last_update` and `freshness_tier` (staleness). W5 noticed this in words ("directly inside the rule's own definition") but did not measure what it does to the score - that is the next cell.
2. **Overlapping windows (type 2) - found, indirect.** `impressions_last_30d` is not a feature, but the 90-day volume features contain that same 30-day window (Spearman 0.95 between the two), so they are near-copies of a label ingredient.
3. **Decision-derived (type 3) - none found.** No health-score, flag or queue columns among the features.
4. **Population selection - disclosed.** Rows are in the table only if they had at least one impression in the trailing 90 days. That choice touches the outcome window, so it belongs in my limitations rather than being left implicit.
5. **How much of the label is a plain function of ordinary columns?** About 56% of `y_high = 1` rows are high through staleness plus visibility alone - no decline signal needed - and a decline-blind rule (position 1-10 OR stale_visible) captures every high row at precision 0.74. So a large share of this task is reading back a rule I wrote.


In [6]:
# 3e. Train-with vs train-without the suspects (grouped 5-fold, out-of-fold) ----
INGREDIENT_N, INGREDIENT_C = ["avg_position", "days_since_last_update"], ["position_tier", "freshness_tier"]
VOLUME_PROXY_N, VOLUME_PROXY_C = ["log_impressions_90d", "log_clicks_90d", "days_with_impressions"], ["impression_tier"]

num_a = [c for c in NUM if c not in INGREDIENT_N];           cat_a = [c for c in CAT if c not in INGREDIENT_C]
num_b = [c for c in num_a if c not in VOLUME_PROXY_N];       cat_b = [c for c in cat_a if c not in VOLUME_PROXY_C]

oof_a, f_a = cv_eval(num_a, cat_a, GroupKFold(5), use_groups=True)
oof_b, f_b = cv_eval(num_b, cat_b, GroupKFold(5), use_groups=True)

# Harness sanity check: deliberately PLANT a leak (trend_pct, a banned column). The score must jump toward 1.0.
df["_planted_leak"] = df["trend_pct"].fillna(0)
oof_p, f_p = cv_eval(NUM + ["_planted_leak"], CAT, GroupKFold(5), use_groups=True)
df.drop(columns="_planted_leak", inplace=True)

ablation = pd.DataFrame([
    summarize("A. W5 feature set as-is (grouped)", f_grp, oof_grp),
    summarize("B. minus 4 label-ingredient columns (position/staleness + their tiers)", f_a, oof_a),
    summarize("C. B minus 4 volume-proxy columns (90d impressions/clicks/days-visible + tier)", f_b, oof_b),
    summarize("PLANTED LEAK: A + trend_pct (harness check, never a real model)", f_p, oof_p),
])[["split", "AUC (mean of folds)", "AUC fold sd", "AUC worst fold", "P@top-10% (mean)", "base rate", "lift @top-10%"]]
print(ablation.to_string(index=False))

# ---- 3f. Top-feature sanity check on one held-out fold (permutation importance, not impurity) ----
tr, te = next(GroupKFold(5).split(df, y, groups))
mm = make_model(NUM, CAT).fit(df.iloc[tr][NUM + CAT], y[tr])
pi = permutation_importance(mm, df.iloc[te][NUM + CAT], y[te], n_repeats=5, random_state=SEED, n_jobs=-1, scoring="roc_auc")
imp = pd.Series(pi.importances_mean, index=NUM + CAT).sort_values(ascending=False)
print("\nTop-6 permutation importance (drop in AUC when shuffled), one held-out fold:")
print(imp.head(6).round(4).to_string())
ingredient_names = set(INGREDIENT_N + INGREDIENT_C)
print("Top-4 that are label-ingredient columns:", [c for c in imp.head(4).index if c in ingredient_names])

RECEIPTS["section3_ablation"] = ablation.to_dict(orient="records")

                                                                         split  AUC (mean of folds)  AUC fold sd  AUC worst fold  P@top-10% (mean)  base rate  lift @top-10%
                                             A. W5 feature set as-is (grouped)                0.946        0.023           0.915             0.967      0.423           2.58
        B. minus 4 label-ingredient columns (position/staleness + their tiers)                0.658        0.059           0.567             0.637      0.423           1.68
C. B minus 4 volume-proxy columns (90d impressions/clicks/days-visible + tier)                0.586        0.057           0.475             0.603      0.423           1.46
               PLANTED LEAK: A + trend_pct (harness check, never a real model)                0.998        0.002           0.994             1.000      0.423           2.69

Top-6 permutation importance (drop in AUC when shuffled), one held-out fold:
freshness_tier            0.1102
position_tier           

**Train-with vs train-without (measured).**

| Model (client-grouped 5-fold, out-of-fold) | AUC | Note |
|---|---|---|
| A. W5 features as-is | 0.946 | the number W5's story rests on |
| B. minus 4 label-ingredient columns | 0.658 | worst fold 0.57 |
| C. B minus 4 volume-proxy columns | 0.586 | worst fold 0.48 (below chance) |
| Planted leak (+ `trend_pct`) | 0.998 | harness check only |

- **The harness works.** Deliberately planting `trend_pct` pushes AUC to about 1.0, so the test would catch a real leak. That gives the other numbers weight.
- **Removing four columns lowered AUC by about 0.29.** This is the pattern the validation skill describes as a confession. The permutation-importance check agrees: the top four features are exactly the four ingredient columns.
- **A fair nuance on naming.** In deployment those four columns *would* be known at scoring time, so this is not "future information" in the strict sense; it is a **circular proxy** - the label is a function of the features. Whether we call it leakage or circularity, the consequence is the same: the 0.95 measures agreement with a rule I wrote, not an ability to find pages that need a refresh.
- **What remains without the ingredients is weak and uneven** (AUC 0.66, worst fold 0.57). That is directional at best.

In [7]:
# ---- 3g. Real failure examples: model C (label ingredients + volume proxies removed), grouped out-of-fold ----
# Choose K = number of true positives so false positives and false negatives are equal BY CONSTRUCTION -
# that way the mix of errors says something about the model rather than about the choice of K.
K = int(y.sum())
picked = np.zeros(len(df), dtype=bool); picked[np.argsort(-oof_b)[:K]] = True
fp = picked & (y == 0);  fn = (~picked) & (y == 1)
page1_only = (p1 & ~sv).values           # 'high' ONLY because of page-1 decay (needs the hidden trend signal)
stale_any = sv.values                    # 'high' because of staleness + visibility (readable without trend)
print(f"K = {K:,} (= number of y_high rows). Precision in top-K: {y[picked].mean():.3f} vs base rate {y.mean():.3f}")
print(f"False positives {fp.sum():,} | false negatives {fn.sum():,}")
print(f"Share of false negatives that were 'high' ONLY via page-1 decay (needs the hidden decline signal): {(fn & page1_only).sum() / fn.sum():.1%}")
print(f"Recall inside the top-K - stale_visible rows: {picked[stale_any & (y == 1)].mean():.1%} | page-1-decay-only rows: {picked[page1_only].mean():.1%}")

cols = ["avg_position", "days_since_last_update", "impressions_last_30d", "ctr", "content_age_days"]
print("\nThree false negatives (actually high, ranked out) - 'via' says which label ingredient made them high:")
ex_fn = df.loc[fn, cols].head(3).copy(); ex_fn["via"] = np.where(page1_only[fn][:3], "page-1 decay", "staleness+visibility")
print(ex_fn.reset_index(drop=True).to_string())
print("\nThree false positives (ranked in, not high):")
print(df.loc[fp, cols].head(3).reset_index(drop=True).to_string())

RECEIPTS["section3_failures"] = {"K": K, "fp": int(fp.sum()), "fn": int(fn.sum()),
                                 "fn_share_page1_decay_only": round(float((fn & page1_only).sum() / fn.sum()), 3),
                                 "recall_stale_visible": round(float(picked[stale_any & (y == 1)].mean()), 3),
                                 "recall_page1_decay_only": round(float(picked[page1_only].mean()), 3)}

K = 12,694 (= number of y_high rows). Precision in top-K: 0.535 vs base rate 0.423
False positives 5,902 | false negatives 5,902
Share of false negatives that were 'high' ONLY via page-1 decay (needs the hidden decline signal): 64.4%
Recall inside the top-K - stale_visible rows: 70.3% | page-1-decay-only rows: 32.4%

Three false negatives (actually high, ranked out) - 'via' says which label ingredient made them high:
   avg_position  days_since_last_update  impressions_last_30d   ctr  content_age_days                   via
0           8.5                      20                   617  0.03               147          page-1 decay
1           7.0                      20                     1  0.00                90          page-1 decay
2          39.8                     103                    85  0.00               238  staleness+visibility

Three false positives (ranked in, not high):
   avg_position  days_since_last_update  impressions_last_30d   ctr  content_age_days
0          36.5

**Real failure examples (illustrations, not patterns - three rows each).**

I used model C (ingredients and volume proxies removed) because that is where errors are informative; model A has almost none. I set K equal to the number of true highs so false positives and false negatives are equal *by construction* - that way the mix of errors says something about the model rather than about K.

- **About 64% of the false negatives are "high" only through page-1 decay**, which needs the decline signal that is banned from the features. The model recovers staleness-driven highs far more often (about 70%) than decline-driven ones (about 32%). This is the structural gap W5 pointed at, now measured.
- **The displayed false negatives look ordinary on visible columns.** Two are page-1 pages updated 20 days ago - nothing in their features says "declining" - and one is a staleness case (103 days, 85 recent impressions).
- **The displayed false positives are highly visible pages** (roughly 2,400-4,200 impressions in 30 days), two at positions 36-44 and one on page 1 but recently updated. With three rows I will only say that the ranking appears to lean on visibility; I would not claim more.
- Precision in the top-K is 0.54 against a 0.42 base rate - a modest, directional lift once the ingredients are gone.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

I re-read my Week-5 notebook looking for sentences that go further than the evidence. Four qualify; the first is my boldest.

| # | What W5 said | What the evidence supports |
|---|---|---|
| 1 (boldest) | "The model's real advantage only shows up once you ask it to rank deeper... Random Forest holds ~0.90 [at K=1,000]" | Measured on one 8-client split, against a base rate of 0.36 and a baseline rule that was never fit to the label. Under 5-fold client grouping AUC is 0.95, but removing four label-ingredient columns lowers it to 0.66 - so the headline largely reflects rule reconstruction. |
| 2 | "Errors skew heavily toward false negatives... a model that's stingy... is the safer failure mode." | With 2,591 true highs and K=1,000, at least 1,591 misses were unavoidable by arithmetic. The skew is largely a consequence of K, so it says little about model caution. |
| 3 | "What actually made them 'high' is the decline signal" (three displayed misses) | Two of three are decline-driven page-1 decay; one is staleness plus visibility, which is readable without any trend. |
| 4 | "Where it earns its keep is in finding the medium-strength combinations (word count, engagement, AI-traffic share, content type)" | Not tested. With ingredient and volume columns removed, measured AUC is 0.59 - a hypothesis to test, not a finding. |

**Boldest sentence, rewritten (#1):**

> In a client-grouped 5-fold check across all 32 clients, the Random Forest's out-of-fold AUC was measured at about 0.95 (worst fold about 0.92) against a 0.42 base rate. This is directional: it largely reflects the model re-reading the columns that define the proxy label, since removing four of them lowered AUC to 0.66. It is decision-support for ordering pages for human review; the proxy label has not been checked against editor-confirmed outcomes.

The code cell below re-verifies claims #2 and #3 against the data, builds all four rewrites with live numbers, and screens them for overclaiming words.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---- Re-checking my own W5 sentences against the data ----
# W5 claim: "Errors skew heavily toward false negatives ... a model that's stingy ... is the safer failure mode" (at K=1,000)
tr5, te5 = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED).split(df, y, groups))
n_pos_test = int(y[te5].sum())
print(f"W5 test set: {len(te5):,} rows, {n_pos_test:,} true 'high'. Ranking only K=1,000 means at least {n_pos_test - 1000:,} 'high' rows MUST fall outside the list.")
print("-> the W5 false-negative surplus (1,688 vs 97) follows largely from the choice of K, so it is weak evidence about how cautious the model is.")

# W5 claim: for the 3 displayed false negatives, 'what actually made them high is the decline signal'
# (matched on the three feature tuples W5 printed: avg_position, days_since_last_update, impressions_last_30d)
w5_rows = [(8.5, 20, 617), (39.8, 103, 85), (6.9, 20, 10)]
key = list(zip(df["avg_position"], df["days_since_last_update"], df["impressions_last_30d"]))
hit = [i for i, k in enumerate(key) if k in set(w5_rows)]
chk = pd.DataFrame({"page1_decay": p1.values[hit], "stale_visible": sv.values[hit]})
chk = chk[chk.index < 3] if len(chk) > 3 else chk
print(f"\nWhy each of the {len(chk)} displayed W5 false negatives is 'high':")
print(chk.rename_axis("example").to_string())
print(f"-> {int(chk['page1_decay'].sum())} of {len(chk)} via decline-dependent page-1 decay; {int(chk['stale_visible'].sum())} of {len(chk)} via staleness plus visibility (readable without any trend).")

# ---- the REWRITTEN claims, with numbers pulled live so the text cannot drift from the evidence ----
auc_A, worst_A = f_grp["auc"].mean(), f_grp["auc"].min()
auc_B, auc_C, worst_C = f_a["auc"].mean(), f_b["auc"].mean(), f_b["auc"].min()
REWRITTEN = {
 "headline": (f"In a client-grouped 5-fold check across all 32 clients, the Random Forest's out-of-fold AUC was measured at {auc_A:.2f} "
              f"(worst fold {worst_A:.2f}) against a {y.mean():.2f} base rate. This is directional: it largely reflects the model re-reading the columns "
              f"that define the proxy label, since removing four of them lowered AUC to {auc_B:.2f}. "
              f"It is decision-support for ordering pages for human review; the proxy label has not been checked against editor-confirmed outcomes."),
 "errors":   (f"At K=1,000 of a {len(te5):,}-row held-out set with {n_pos_test:,} proxy-'high' rows, at least {n_pos_test - 1000:,} misses were unavoidable "
              f"by construction, so the false-negative surplus does not by itself show how cautious the model is."),
 "examples": (f"Of the three displayed misses, {int(chk['page1_decay'].sum())} were 'high' through page-1 decay (which depends on the decline signal the model never sees) "
              f"and {int(chk['stale_visible'].sum())} through staleness plus visibility; the single explanation offered in W5 fits only the former."),
 "secondary":(f"Whether word count, engagement or AI-traffic share add information beyond the label's own ingredients was not tested in W5. "
              f"With the ingredient and volume columns removed, measured AUC was {auc_C:.2f} (worst fold {worst_C:.2f}), so any such contribution is a hypothesis to test, not a finding."),
}
RISKY = ["prove", "proves", "proved", "cause", "causes", "caused", "guarantee", "guarantees", "always", "best", "significantly", "will improve", "clearly shows"]
SAFE = ["measured", "observed", "directional", "decision-support", "hypothesis", "proxy"]
for k, v in REWRITTEN.items():
    words = set(v.lower().replace(",", " ").replace(".", " ").replace(";", " ").split())
    hits = [w for w in RISKY if w in words]
    safe = [w for w in SAFE if w in v.lower()]
    print(f"[{k}] risky words: {hits if hits else 'none'} | safe-language markers used: {safe}")
    print("    " + v)

# ---- receipts: the committed metrics file every number above traces back to ----
RECEIPTS["section4_claims"] = {"k1000_min_unavoidable_misses": n_pos_test - 1000,
                               "w5_displayed_fn_via_page1_decay": int(chk["page1_decay"].sum()),
                               "w5_displayed_fn_via_staleness": int(chk["stale_visible"].sum()),
                               "rewritten": REWRITTEN}
os.makedirs("work/outputs", exist_ok=True)
with open("work/outputs/w06_validation_audit_metrics.json", "w") as fh:
    json.dump({"seed": SEED, "rows": int(len(df)), "clients": int(df["client_id"].nunique()),
               "target_base_rate": round(float(y.mean()), 3), **RECEIPTS}, fh, indent=2, default=str)
print("\nWrote work/outputs/w06_validation_audit_metrics.json")


W5 test set: 7,115 rows, 2,591 true 'high'. Ranking only K=1,000 means at least 1,591 'high' rows MUST fall outside the list.
-> the W5 false-negative surplus (1,688 vs 97) follows largely from the choice of K, so it is weak evidence about how cautious the model is.

Why each of the 3 displayed W5 false negatives is 'high':
         page1_decay  stale_visible
example                            
0               True          False
1              False           True
2               True          False
-> 2 of 3 via decline-dependent page-1 decay; 1 of 3 via staleness plus visibility (readable without any trend).
[headline] risky words: none | safe-language markers used: ['measured', 'directional', 'decision-support', 'proxy']
    In a client-grouped 5-fold check across all 32 clients, the Random Forest's out-of-fold AUC was measured at 0.95 (worst fold 0.92) against a 0.42 base rate. This is directional: it largely reflects the model re-reading the columns that define the proxy label, s

**What I will change in the capstone write-up (limitations, stated plainly):**

- The target is a **proxy** built from a rule; no editor-confirmed outcome exists, so scores are agreement with the proxy.
- The strongest features are **ingredients of that proxy**; the score should be read with the ablation table beside it.
- Rows are kept only if they had impressions in the trailing 90 days (**population choice**).
- The starter data is **one snapshot**: no time-aware validation was possible, so nothing here says how the model behaves on next month's pages.
- The ranked queue is **decision-support for human review**, not an automated refresh decision.

**Where to take this next (not done here):** rebuild the label so that features come strictly before an outcome window - the warehouse tables in `flyrank-data` support a past-to-future design on a mid-panel month with the final month kept sealed. That is the version of this model that could be tested against something it cannot read off the page.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.